# 🔹 Notebook 6 — Modelli di Dati KG in Python

**Obiettivo**: Implementare e confrontare RDF (rdflib) e Property Graph (NetworkX/Neo4j) con gli stessi dati.

Corrisponde al capitolo **Modelli di Dati per i Knowledge Graph**.


## Setup

In [ ]:
# Se non installato: pip install rdflib networkx neo4j matplotlib
import json
try:
    from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef
    print('rdflib OK')
except: print('pip install rdflib')

try:
    import networkx as nx
    print('networkx OK')
except: print('pip install networkx')

print('Setup completo.')

## 6.1 — Modello RDF con rdflib

In [ ]:
# Creiamo un grafo RDF con le stesse entita' del libro
from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef, XSD

g = Graph()
EX = Namespace('http://example.org/')
FOAF = Namespace('http://xmlns.com/foaf/0.1/')
g.bind('ex', EX)
g.bind('foaf', FOAF)

# Aggiungere triple
g.add((EX.alice, RDF.type, FOAF.Person))
g.add((EX.alice, FOAF.name, Literal('Alice Smith')))
g.add((EX.alice, FOAF.age, Literal(30, datatype=XSD.integer)))
g.add((EX.alice, FOAF.knows, EX.bob))

g.add((EX.bob, RDF.type, FOAF.Person))
g.add((EX.bob, FOAF.name, Literal('Bob Jones')))
g.add((EX.bob, FOAF.age, Literal(28, datatype=XSD.integer)))

g.add((EX.techcorp, RDF.type, EX.Company))
g.add((EX.techcorp, FOAF.name, Literal('TechCorp')))
g.add((EX.techcorp, EX.industry, Literal('Software')))

g.add((EX.alice, EX.works_at, EX.techcorp))
g.add((EX.bob, EX.works_at, EX.techcorp))

print(f'Triple nel grafo RDF: {len(g)}')
print()
print('Serializzazione Turtle:')
print(g.serialize(format='turtle'))

## 6.2 — Query SPARQL su grafo locale

In [ ]:
# SPARQL SELECT: chi lavora dove?
results = g.query("""
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX ex: <http://example.org/>
    SELECT ?name ?company WHERE {
        ?person a foaf:Person ;
                foaf:name ?name ;
                ex:works_at ?comp .
        ?comp foaf:name ?company .
    }
""")

print('Chi lavora dove (SPARQL):')
for row in results:
    print(f'  {row.name} -> {row.company}')

In [ ]:
# SPARQL ASK: Alice conosce Bob?
result = g.query("""
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX ex: <http://example.org/>
    ASK { ex:alice foaf:knows ex:bob . }
""")
print(f'Alice conosce Bob? {bool(result)}')

## 6.3 — Serializzazione: Turtle, JSON-LD, RDF/XML

In [ ]:
# Confronto formati
for fmt in ['turtle', 'json-ld', 'xml']:
    output = g.serialize(format=fmt)
    lines = output.strip().split('\n')
    print(f'\n=== {fmt.upper()} ({len(lines)} righe) ===')
    for line in lines[:8]:
        print(f'  {line}')
    if len(lines) > 8:
        print(f'  ... ({len(lines)-8} altre righe)')

## 6.4 — Stesso grafo come Property Graph (NetworkX)

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Nodi con proprieta'
G.add_node('alice', labels=['Person'], name='Alice Smith', age=30)
G.add_node('bob', labels=['Person'], name='Bob Jones', age=28)
G.add_node('techcorp', labels=['Company'], name='TechCorp', industry='Software')

# Archi con label e proprieta'
G.add_edge('alice', 'techcorp', label='WORKS_AT', since=2020)
G.add_edge('bob', 'techcorp', label='WORKS_AT', since=2019)
G.add_edge('alice', 'bob', label='KNOWS', since=2019)

print(f'Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}')
print()
for n, data in G.nodes(data=True):
    print(f'  [{data.get("labels",[])}] {data.get("name",n)}')
for u, v, data in G.edges(data=True):
    print(f'  {u} --[{data["label"]}]--> {v}')

## 6.5 — Pattern matching su Property Graph

In [ ]:
# 'Trova chi lavora a TechCorp' - in Property Graph e' navigazione diretta
works_at_techcorp = [u for u, v, d in G.edges(data=True)
                     if v == 'techcorp' and d.get('label') == 'WORKS_AT']
print('Chi lavora a TechCorp (PG):', works_at_techcorp)

# Shortest path
path = nx.shortest_path(G, 'alice', 'techcorp')
print(f'Path alice -> techcorp: {path}')

# Degree centrality (hub analysis)
centrality = nx.degree_centrality(G)
for node, score in sorted(centrality.items(), key=lambda x: -x[1]):
    print(f'  {node}: {score:.2f}')

## 6.6 — Classe KnowledgeGraph custom

In [ ]:
class KnowledgeGraph:
    def __init__(self):
        self.nodes = {}
        self.edges = {}
        self.index = {}  # label -> [node_ids]
    
    def add_node(self, nid, labels, props):
        self.nodes[nid] = {'id': nid, 'labels': labels, 'properties': props}
        for l in labels:
            self.index.setdefault(l, []).append(nid)
    
    def add_edge(self, eid, src, tgt, label, props=None):
        self.edges[eid] = {'id':eid, 'from':src, 'to':tgt, 'label':label, 'properties':props or {}}
    
    def find_by_label(self, label):
        return [self.nodes[nid] for nid in self.index.get(label, [])]
    
    def get_neighbors(self, nid, rel=None):
        return [self.nodes[e['to']] for e in self.edges.values()
                if e['from'] == nid and (rel is None or e['label'] == rel)]
    
    def stats(self):
        return f'Nodi: {len(self.nodes)}, Archi: {len(self.edges)}, Labels: {list(self.index.keys())}'

kg = KnowledgeGraph()
kg.add_node('alice', ['Person'], {'name': 'Alice', 'age': 30})
kg.add_node('techcorp', ['Company'], {'name': 'TechCorp', 'industry': 'Software'})
kg.add_edge('e1', 'alice', 'techcorp', 'WORKS_AT', {'since': 2020})

print(kg.stats())
print('Persone:', [n['properties']['name'] for n in kg.find_by_label('Person')])
print('Alice lavora a:', [n['properties']['name'] for n in kg.get_neighbors('alice', 'WORKS_AT')])

## 6.7 — Confronto diretto: stessa domanda, due modelli

In [ ]:
print('=== Domanda: Chi lavora a TechCorp? ===')
print()

# RDF/SPARQL
results = g.query("""
    PREFIX ex: <http://example.org/> PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    SELECT ?name WHERE { ?p ex:works_at ex:techcorp . ?p foaf:name ?name . }
""")
print('RDF/SPARQL:', [str(r.name) for r in results])

# Property Graph/NetworkX
pg_result = [G.nodes[u]['name'] for u, v, d in G.edges(data=True)
             if v == 'techcorp' and d.get('label') == 'WORKS_AT']
print('PG/NetworkX:', pg_result)

# Custom KG class
print('Custom KG:', [n['properties']['name'] for n in kg.find_by_label('Person')
                     if kg.get_neighbors(n['id'], 'WORKS_AT')])